In [24]:
# Read text file
with open("text.txt", "r", encoding="utf-8") as f:
    text = f.read().lower()

In [25]:
# Check first few characters
print(text[:500])

﻿
project gutenberg's the adventures of sherlock holmes, by arthur conan doyle

this ebook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  you may copy it, give it away or
re-use it under the terms of the project gutenberg license included
with this ebook or online at www.gutenberg.net


title: the adventures of sherlock holmes

author: arthur conan doyle

release date: november 29, 2002 [ebook #1661]
last updated: may 20, 2019

language: english

charac


In [26]:
import re
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer


# remove unnecessary symbols/punctuation
text = re.sub(r'[^a-zA-Z\s]', '', text)

# remove unnecessary line break
text = text.replace("\n", " ")
text = text.replace("\r", " ")

# check cleaned text
print(text[:500])

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

# total unique words
total_words = len(tokenizer.word_index) + 1

print("Total unique words:", total_words)


 project gutenbergs the adventures of sherlock holmes by arthur conan doyle  this ebook is for the use of anyone anywhere at no cost and with almost no restrictions whatsoever  you may copy it give it away or reuse it under the terms of the project gutenberg license included with this ebook or online at wwwgutenbergnet   title the adventures of sherlock holmes  author arthur conan doyle  release date november   ebook  last updated may    language english  character set encoding utf   start of th
Total unique words: 8601


In [27]:
input_sequences = []

tokenized_sequence = tokenizer.texts_to_sequences([text])[0]

sequence_length = 20

for i in range(sequence_length, len(tokenized_sequence)):
    seq = tokenized_sequence[i-sequence_length:i+1]
    input_sequences.append(seq)

input_sequences = np.array(input_sequences)

print(input_sequences.shape)

(107410, 21)


In [28]:
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print(X.shape)
print(y.shape)

(107410, 20)
(107410,)


In [29]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Input

model = Sequential()

# Input: 20 previous words
model.add(Input(shape=(X.shape[1],)))

# Convert word IDs into dense vectors
model.add(Embedding(input_dim=total_words,output_dim=100))

# First LSTM layer
model.add(LSTM(150, return_sequences=True))

# Second LSTM layer
model.add(LSTM(150))

# Reduce overfitting
model.add(Dropout(0.1))

# Predict next word from vocabulary
model.add(Dense(total_words,activation='softmax'))

# Compile
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 20, 100)        │       860,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 20, 150)        │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 150)            │       180,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 150)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 8601)           │     1,298,751 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,490,051 (9.50 MB)

 Trainable params: 2,490,051 (9.50 MB)

 Non-trainable params: 0 (0.00 B)

In [30]:
# from tensorflow.keras.callbacks import EarlyStopping

# early_stop = EarlyStopping(
#     monitor='val_loss',
#     patience=5,
#     restore_best_weights=True
# )

history = model.fit(
    X,
    y,
    epochs=100,
    batch_size=128,
    validation_split=0.2,

    verbose=1
)

Epoch 1/100
672/672 ━━━━━━━━━━━━━━━━━━━━ 11s 14ms/step - accuracy: 0.0569 - loss: 6.6031 - val_accuracy: 0.0602 - val_loss: 6.5602
Epoch 2/100
672/672 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.0720 - loss: 6.1087 - val_accuracy: 0.0720 - val_loss: 6.4257
Epoch 3/100
672/672 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.0860 - loss: 5.8757 - val_accuracy: 0.0822 - val_loss: 6.4031
Epoch 4/100
672/672 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.1001 - loss: 5.7016 - val_accuracy: 0.0926 - val_loss: 6.3572
Epoch 5/100
672/672 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - accuracy: 0.1113 - loss: 5.5526 - val_accuracy: 0.1019 - val_loss: 6.3607
Epoch 6/100
672/672 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.1206 - loss: 5.4166 - val_accuracy: 0.1072 - val_loss: 6.3488
Epoch 7/100
672/672 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.1285 - loss: 5.2942 - val_accuracy: 0.1105 - val_loss: 6.3737
Epoch 8/100
672/672 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.1343 - loss: 5.1830 -

In [31]:
model.save("100model.keras")

In [32]:
from tensorflow.keras.models import load_model
import numpy as np

# Load model
model = load_model("100model.keras")

In [33]:
def predict_next_words(seed_text, num_words):

    for _ in range(num_words):

        # Convert input text to tokens
        token_list = tokenizer.texts_to_sequences([seed_text])[0]

        # Last 20 words hi lo (same as training)
        token_list = token_list[-20:]

        # Pad if length < 20
        token_list = np.pad(
            token_list,
            (20-len(token_list), 0),
            mode='constant'
        )

        # Add batch dimension
        token_list = np.array([token_list])

        # Predict
        predicted = model.predict(token_list, verbose=0)

        # Highest probability word index
        predicted_word_index = np.argmax(predicted)

        # Convert index → word
        for word, index in tokenizer.word_index.items():
            if index == predicted_word_index:
                seed_text += " " + word
                break

    return seed_text

In [42]:
print(
    predict_next_words(
        "my name",
        2

    )
)

my name is a


In [43]:
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [44]:
import json

config = {
    "sequence_length": sequence_length
}

with open("config.json", "w") as f:
    json.dump(config, f)